In [50]:

import os, io, time, json, hashlib, pathlib, sys
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import re
import matplotlib.pyplot as plt
from urllib.parse import urlparse
from datetime import datetime, timedelta
from config import *
# from functions1 import *
from functions2 import *

def build_returns_matrix_in_chf(
    holdings: list[dict],
    lookback_days: int = 725,
    max_age: int = 24,
    no_fx: bool = False,
    usd_shift: bool = False,
    DEBUG: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    """
    Build CHF daily returns matrix for the provided holdings.

        holdings: list of dicts with tickers:
      - name: row/column label in outputs
      - ticker: EODHD tickerbol (e.g., 'SWDA.LON', 'IBM')
      - ccy: base currency of the asset price series (GBP/USD/EUR/CHF)
      - gbx: bool; if True, divide close by 100.0 (LSE pence)
      - position: number of shares held (float)

        Returns:
      rets_df: DataFrame of CHF log returns, T x N
      prices_df: DataFrame of CHF closes, T x N
      weights: Series aligned to columns in rets_df
    """

    params = {
                    'from': START, 
                    'to': today,
                    'api_token': EOD_API    
                }
    url_fx = f'https://eodhd.com/api/eod/'

    # -------------------------------------------------------
    # --------------------------------------------------------
    
    fx_map = make_fx_map(url_fx, holdings, params, max_age, no_fx, usd_shift)
    if DEBUG:
        print(f">>>FX keys loaded: {list(fx_map.keys())}")
    print('-------------fetching assets-------------')
    # Build CHF close series per asset

    url = f'https://eodhd.com/api/eod/'

    # --------------------------------------------------------
    def make_assets_df(url:str, holding: dict, fx_map: dict,params: dict, max_age: int) -> pd.DataFrame:
        '''
        loop through holdings
            create local series -> add to holdings local dataframe -> 
            create chf close series -> add to dataframe
        '''
        assets_close_local_df = pd.DataFrame()
        assets_close_chf_df = pd.DataFrame()
        for h in holdings:
            name = h['name']
            asset_close_local_s = fetch_asset_series(url, h, fx_map, params=params, max_age=max_age, lookback_days=lookback_days)
            assets_close_local_df[name] = asset_close_local_s
            # print(f'asset_closer_local type: {type(asset_close_local_s)}')

            # --------------------------------------------------------    # 
            def create_asset_close_chf_s(asset_close_local_s: pd.Series, holding: dict, fx_map: dict, no_fx: bool) -> pd.DataFrame:
                '''
                take the holding local series and return a chf series
                this is for risk calc, so ERNS should not have FX applied, otherwise its takes on currency vol - really
                '''
                ccy = holding["ccy"].upper()
                chf_close_s = pd.Series()
                last_asset_close = {}
                gbx   = holding.get("gbx",False)
                assert isinstance(gbx, bool), f"gbx should be bool, got {type(gbx)}"
                if gbx:
                    asset_close_local_s= asset_close_local_s/ 100.0
                assert float(asset_close_local_s.median()) < 1000, f"Looks like pence still (median {float(asset_close_local_s.median())}); did you divide by 100?"
                if 'VEU.US' in name:
                    asset_close_chf_s= asset_close_local_s* 0.9
                    print('VEU (XWMX proxy) * 0.9 to remove EM element')
                last_asset_close[name] = asset_close_local_s.iloc[-1]

                # DONT CONVERT FOR CHF CASH, OR IF NO_FX FLAG SET, OR IF RISK_FX SET TO 'NONE'
                if (ccy == "CHF") or no_fx or holding.get('risk_fx', '').lower() == 'none':
                    print("    Skipping FX conversion for", name)
                    asset_close_chf_s = asset_close_local_s.rename(name)
                else:
                    fx = fx_map[f'{ccy}CHF']
                    # Align FX to price dates and forward-fill FX only (never prices)
                    before = fx.reindex(asset_close_local_s.index)
                    fx_aligned = before.ffill()
                    filled = int(before.isna().sum() - fx_aligned.isna().sum())
                    if DEBUG and filled > 0:
                        print(f"    {name}: FX ffill filled {filled} missing FX points on price dates")
                    asset_close_chf_s = (asset_close_local_s * fx_aligned).dropna().rename(name)
                return asset_close_chf_s
            # --------------------------------------------------------

            asset_close_chf_s = create_asset_close_chf_s(asset_close_local_s, h, fx_map, no_fx)
            assets_close_chf_df[name] = asset_close_chf_s
        return assets_close_local_df, assets_close_chf_df

    assets_close_local_df, assets_close_chf_df = make_assets_df(url, holdings, fx_map, params, max_age)        

    
    print(f"assets local close, {assets_close_local_df}")
    print(f"assets chf close, {assets_close_chf_df}")

    # After assets_close_chf_df is built, before prices_df/rets_df
    hedged_cash = [
        h['name'] for h in holdings
        if h.get('type','').lower() == 'cash' and h.get('risk_fx','').lower() == 'none'
    ]
    for n in hedged_cash:
        if n in assets_close_chf_df.columns:
            assets_close_chf_df[n] = 1.0


    # Align on common dates and restrict to lookback window
    prices_df = pd.DataFrame(assets_close_chf_df).dropna()
   
    if DEBUG and not prices_df.empty:
        print(
            f">>>Common window: {prices_df.index.min().date()} → {prices_df.index.max().date()} "
            f"({len(prices_df)} rows before tail)"
        )

    prices_df = prices_df.tail(lookback_days)
    rets_df = np.log(prices_df / prices_df.shift(1)).dropna()

    if prices_df.shape[0] < (lookback_days * 0.73):
        print(
            f"After alignment only {prices_df.shape[0]} rows remain "
            f"(expected {lookback_days}). Data source may not have full history."
        )
    
    if rets_df.isna().any().any():
        raise ValueError("NaNs remained in returns after shift/drop; check data alignment.")
    if (prices_df <= 0).any().any():
        raise ValueError("Non-positive prices encountered; check source data.")
    
    # for h in holdings:
    #     name = h['name']
    #     if h.get("type", "").lower() == "cash":
    #         if h.get('ccy', '').upper() != 'CHF':
    #             chfval = h['amount'] * fx_map.get(f"{h.get('ccy','').upper()}CHF", pd.Series([np.nan])).iloc[-1]
    #             chf_values[name] = chfval
    #         else:
    #             chf_values[name] = h['amount']
    #         continue
    #     last_price = assets_close_chf_df[name].iloc[-1]
    #     chf_values[name] = h['position'] * last_price
    # print(f'CHF values per asset: {chf_values}')

    asof = prices_df.index[-1]
    chf_values = {}

    for h in holdings:
        name = h['name']
        typ = h.get('type', '').lower()
        ccy = h.get('ccy', '').upper()

        if typ == 'cash':
            if ccy == 'CHF':
                chf_values[name] = float(h['amount'])
            else:
                pair = f"{ccy}CHF"
                fx = fx_map.get(pair, pd.Series(dtype=float))
                if fx.empty:
                    raise KeyError(f"Missing FX series {pair} for cash {name}")
                # Align FX to portfolio as-of date and ffill FX only
                last_fx = fx.reindex([asof]).ffill().iloc[-1]
                if np.isnan(last_fx):
                    raise ValueError(f"FX {pair} has no value on/before {asof} for cash {name}")
                chf_values[name] = float(h['amount']) * float(last_fx)
            continue

        # Non-cash assets: always value in CHF at the common as-of date
        if ccy == 'CHF':
            last_local = assets_close_local_df[name].reindex([asof]).iloc[-1]
            last_chf = float(last_local)
        elif h.get('risk_fx', '').lower() == 'none':
            # Hedged in risk (kept in local for returns), but still valued in CHF
            last_local = assets_close_local_df[name].reindex([asof]).iloc[-1]
            pair = f"{ccy}CHF"
            fx = fx_map.get(pair, pd.Series(dtype=float))
            if fx.empty:
                raise KeyError(f"Missing FX series {pair} for asset {name}")
            last_fx = fx.reindex([asof]).ffill().iloc[-1]
            if np.isnan(last_fx):
                raise ValueError(f"FX {pair} has no value on/before {asof} for asset {name}")
            last_chf = float(last_local) * float(last_fx)
        else:
            # Already CHF-converted series (and aligned) – use the as-of price
            last_chf = float(assets_close_chf_df[name].reindex([asof]).iloc[-1])

        chf_values[name] = float(h['position']) * last_chf

    

    total_val = sum(chf_values.values())
    print(f'LOOKBACK DAYS/REGIME: {lookback_days}')
    print(f"Total portfolio value (CHF): {total_val:.2f}")


    # Weights (CHF)
    weights = pd.Series()
    for h in holdings:
        name = h["name"]
        value = float(chf_values[h["name"]])
        weight = value / total_val
        last = 0
        if 'type' in h:
            last = assets_close_chf_df[name].iloc[-1]
        else:
            last = assets_close_local_df[name].iloc[-1]

        print(f"    {name}: value {value:.2f} CHF, weight: {weight:.4%} last: {last:.2f}")
        weights[name] = weight
    

    if not np.isclose(weights.sum(), 1.0, atol=1e-6):
        raise ValueError(f"Weights must sum to 1. Got {weights.sum():.6f}" "check postions input in holdings.")
    return rets_df, prices_df, weights



def portfolio_risk(rets_df: pd.DataFrame, weights: pd.Series) -> dict:
    """
    Compute annualized vols, correlation, covariance, portfolio vol,
    marginal risk contribution (MRC), and percent risk contribution (PRC).
    """
    # Annualized stats
    cov_daily = rets_df.cov()
    cov_annual = cov_daily * 252.0
    vol_ann = rets_df.std() * np.sqrt(252.0)
    corr = rets_df.corr()

    # Align weights
    w = weights.reindex(rets_df.columns).astype(float)
    Sigma_w = cov_annual @ w
    port_var = float(w @ Sigma_w)
    port_vol = float(np.sqrt(port_var)) if port_var > 0 else 0.0

    # Contributions
    mrc = Sigma_w / port_vol if port_vol > 0 else Sigma_w * 0.0
    prc = (w * Sigma_w) / port_var if port_var > 0 else w * 0.0

    summary = pd.DataFrame({
        "Weight": w,
        "Vol_1Y_CHF": vol_ann,
        "MRC": mrc,           # ∂σ_p/∂w_i (absolute marginal contribution)
        "PRC_%": prc * 100.0  # percent contribution to total variance (sums ~100%)
    }).sort_values("PRC_%", ascending=False)

    return {
        "port_vol": port_vol,
        "cov_annual": cov_annual,
        "corr": corr,
        "vol_ann": vol_ann,
        "mrc": mrc,
        "prc": prc,
        "summary": summary,
    }

In [51]:
holdings0 = [
    {"name":"Unilever", "ticker":"ULVR.LON", "ccy":"GBP", "gbx":True,  "value_chf": 25000},
    {"name":"Shell",    "ticker":"SHEL.LON", "ccy":"GBP", "gbx":True,  "value_chf": 13000},
    {"name":"NatWest",  "ticker":"NWG.LON",  "ccy":"GBP", "gbx":True,  "value_chf":  5000},
    {"name":"Barclays", "ticker":"BARC.LON", "ccy":"GBP", "gbx":True,  "value_chf":  5000},
    {"name":"Tesco",    "ticker":"TSCO.LON", "ccy":"GBP", "gbx":True,  "value_chf":  5000},
    {"name":"SWDA",     "ticker":"SWDA.LON", "ccy":"GBP", "gbx":True,  "value_chf":  12000},
    {"name":"EMIM",     "ticker":"EMIM.LON", "ccy":"GBP", "gbx":True,  "value_chf":  8000},
    {"name":"IBM",      "ticker":"IBM",      "ccy":"USD", "gbx":False, "value_chf":  4000},
    {"name":"ERNS",     "ticker":"ERNS.LON", "ccy":"GBP", "gbx":True,  "value_chf":  5000},
]
model = [
    {"name":"EMIM",     "ticker":"EMIM.LSE", "ccy":"GBP", "gbx":True, "position": 120},
    # {"name":"ERNS",     "ticker":"ERNS.LSE", "ccy":"GBP", "gbx":False, "position": 100, 'risk_fx': "none"},
    # {"name":"CASH",     "ticker":"ERNS.LSE", "ccy":"GBP", "gbx":False, "position": 119, 'risk_fx': "none"},
    # {"name":"IBM",     "ticker":"IBM.US", "ccy":"USD", "gbx":False, "position": 4},

    {"name":"SGLN",      "ticker":"SGLN.LSE",      "ccy":"GBP", "gbx":True, "position": 50},

    {"name":"VUAG",     "ticker":"VUAG.LSE", "ccy":"GBP", "gbx":False, "position": 43},
    {"name":"WSML",     "ticker":"WSML.LSE", "ccy":"USD", "gbx":False, "position": 365},
    # {"name": "NUCL", "ticker": "NUCL.SW", "ccy": "CHF", "gbx":False, "position": 59},
    # {"name": "URNM", "ticker": "URNM.LSE", "ccy": "USD", "gbx":False, "position": 210},
    {"name": "YCA", "ticker": "YCA.LSE", "ccy": "GBP", "gbx":True, "position": 207},
    # {"name":"XMWX",     "ticker":"XMWX.LSE", "ccy":"GBP", "gbx":False, "position": 125,},
    {"name":"VEU",     "ticker":"VEU.US", "ccy":"USD", "gbx":False, "position": 59,},
    # {"name":"INRG",     "ticker":"INRG.LSE", "ccy":"GBP", "gbx":True, "position": 314,},
    # {"name":"IH2O",     "ticker":"IH2O.LSE", "ccy":"GBP", "gbx":True, "position": 36,},
    # {"name":"REMX",     "ticker":"REMX.SW", "ccy":"CHF", "gbx":False, "position": 55,},
    # {"name":"IFRA",     "ticker":"IFRA.US", "ccy":"USD", "gbx":False, "position": 20,},

    {"name": "CASH_CHF", "type": "cash", "ccy": "CHF", "amount": 1000},
    {"name": "CASH_GBP", "type": "cash", "ccy": "GBP", "amount": 4000},
    {"name": "CASH_USD", "type": "cash", "ccy": "USD", "amount": -2584},

]
IBKR =[
    {"name":"EMIM",     "ticker":"EMIM.LSE", "ccy":"GBP", "gbx":True,  "position": 100},
    # {"name":"XMWX",     "ticker":"XMWX.LSE", "ccy":"GBP", "GPB_exposure": 12.6 "gbx":False, "position": 125,},
    {"name":"ERNS",     "ticker":"ERNS.LSE", "ccy":"GBP", "GBP_exposure": 1,"gbx":False, "position": 119.35, 'risk_fx': "none"},
    {"name":"VEU",     "ticker":"VEU.US", "ccy":"USD", "GBP_exposure": 12.6, "gbx":False, "position": 63,},
    {"name":"IBM",     "ticker":"IBM.US", "ccy":"USD", "gbx":False, "position": 4},
    {"name":"SGLN",      "ticker":"SGLN.LSE","ccy":"GBP", "GBP_exposure": 1,"gbx":True, "position": 40},
    {"name":"VUAG",   "ticker":"VUAG.LSE", "ccy":"GBP",  "USD_exposure": 1, "gbx":False, "position": 28},
    {"name":"WSML",     "ticker":"WSML.LSE", "ccy":"USD", "USD_exposure": 0.58, "gbx":False, "position": 313},
    {"name": "CASH_CHF", "type": "cash", "ccy": "CHF", "amount": 4548},
    {"name": "CASH_GBP", "type": "cash", "ccy": "GBP", "amount": 4000},
    {"name": "CASH_USD", "type": "cash", "ccy": "USD", "amount": -2584},
]
holdings = IBKR
MAX_AGE = 12
LOOKBACK_DAYS = 756
DEBUG = False
START = '2020-01-01'

rets_df, prices_df, w = build_returns_matrix_in_chf(holdings, lookback_days=LOOKBACK_DAYS, max_age=MAX_AGE, no_fx=False, usd_shift=False, DEBUG=DEBUG)

# print(f'rets_df: {rets_df.tail()}')
# print(f'prices_df: {prices_df.tail()}')

risk = portfolio_risk(rets_df, w)

# print(rets_df.tail())
print("Portfolio σ (annualized, CHF): {:.2%}".format(risk["port_vol"]))
print(" CORE EQUITY BAND = 15 - 30%, DIVERSIFIERS = 5 - 12%")
print(risk["summary"].round({"Weight":3,"Vol_1Y_CHF":3,"MRC":3,"PRC_%":1, }))
# Optional:
print(f'CORRELATION:')
print(f'{risk["corr"].round(2)}')
print(f'COVARIANCE:')
print(f'{risk["cov_annual"]}')

-------------fetching currencies-------------
-------------fetching assets-------------
    Skipping FX conversion for ERNS
    Skipping FX conversion for CASH_CHF
assets local close,                EMIM      ERNS      VEU       IBM     SGLN     VUAG    WSML  \
Date                                                                         
2020-01-02  2331.25   86.9091  46.1080  100.5657  2271.25  43.9075  5.5310   
2020-01-03  2322.50   86.8875  45.5236   99.7637  2316.00  44.0250  5.5165   
2020-01-06  2291.00   86.8745  45.6083   99.5855  2323.00  43.7775  5.5075   
2020-01-07  2301.50   86.9091  45.5406   99.6523  2341.00  44.0875  5.5150   
2020-01-08  2305.75   86.8572  45.6507  100.4841  2343.00  44.2575  5.5070   
...             ...       ...      ...       ...      ...      ...     ...   
2025-09-15  3151.00  101.7000  71.2474  256.2400  5236.00  93.1550  8.7880   
2025-09-16  3154.00  101.7200  71.2872  257.5200  5238.00  92.7400  8.7350   
2025-09-17  3179.00  101.7900  71.23

In [50]:
def eod_search(quey: str, token: str):
    import requests, pandas as pd
    url = f"https://eodhd.com/api/search/{quey}?api_token={token}&fmt=json"
    r = requests.get(url, timeout=30); r.raise_for_status()
    hits = r.json()
    # Return a small table to pick from
    return pd.DataFrame([{
        "code": h.get("Code"),
        "exchange": h.get("Exchange"),
        "name": h.get("Name"),
        "currency": h.get("Currency"),
        "type": h.get("Type"),
        "startdate": h.get("StartDate"),
        # earliet date

    } for h in hits])

# Usage:
df = eod_search("EMIM", EOD_API)
# pick the line with the longest available history (often XETRA/LSE/SIX)
print(df)

   code exchange                                           name currency type  \
0  XUSE       AS  iShares MSCI World ex-USA UCITS ETF USD (Acc)      USD  ETF   
1  XUSE      LSE  iShares MSCI World ex-USA UCITS ETF USD (Acc)      GBP  ETF   

  startdate  
0      None  
1      None  


## GET THE EARLIEST DATE ##

In [10]:
START = '2020-01-02'
ticker = f'URNM.LSE'
params = {
    'from': START, 
    'to': today,
    'api_token': EOD_API    
}
url_fx = f'https://eodhd.com/api/eod/{ticker}'

# Fetch EODHD daily FX and build a Series
fx_df = fetch_csv_robust(url_fx, params=params, ticker=ticker, max_age=24)

arse
VEU.US - using cached data


## Equity correlation drift check
We’ll:
- compute a 60-day rolling average pairwise correlation across the equity ETFs in your `rets_df`.
- show the 1-year view you’re using now (limited by `XMWX`).
- also compute a 3-year view excluding `XMWX` (if data exist), to see whether the increase is structural or just recent.

In [23]:
# 1) Current 1-year view (already in rets_df)
# Pick equity ETFs present in rets_df
all_cols = list(rets_df.columns)
EQUITY_LIKE = [c for c in ["EMIM","VUAG","WSML","XMWX","IBM"] if c in all_cols]

if len(EQUITY_LIKE) >= 2:
    rets_eq_1y = rets_df[EQUITY_LIKE]
    # 60D rolling average of pairwise correlation (off-diagonal mean)
    def offdiag_mean(corr_m):
        if corr_m.shape[0] < 2:
            return np.nan
        n = corr_m.shape[0]
        return (corr_m.values.sum() - n) / (n*(n-1))

    roll_avg_corr_1y = (
        rets_eq_1y.rolling(60).corr().groupby(level=0).apply(lambda c: offdiag_mean(c))
    )

    print("1Y rolling(60) avg pairwise equity corr (tail):")
    print(roll_avg_corr_1y.dropna().tail(10))
else:
    print("Not enough equity-like tickers to compute 1Y pairwise correlation.")

# 2) Longer 3Y view excluding XMWX (if data available): re-fetch or extend window is out of scope here,
# but we can approximate by checking if older data exist in prices_df; if not, we demonstrate exclusion-only.
try:
    # If you want to explicitly exclude XMWX to avoid its shorter history limiting the window
    EQUITY_NO_XMWX = [c for c in EQUITY_LIKE if c != "XMWX"]
    if len(EQUITY_NO_XMWX) >= 2:
        # Use available rets_df (1Y). For a true 3Y view, rerun build_returns_matrix_in_chf with lookback_days=756.
        rets_eq_ex = rets_df[EQUITY_NO_XMWX]
        roll_avg_corr_ex = (
            rets_eq_ex.rolling(60).corr().groupby(level=0).apply(lambda c: offdiag_mean(c))
        )
        print("Ex-XMWX rolling(60) avg pairwise equity corr (tail):")
        print(roll_avg_corr_ex.dropna().tail(756))
    else:
        print("Not enough non-XMWX equity-like tickers to compute ex-XMWX correlation.")
except Exception as e:
    print("Correlation analysis note:", e)


1Y rolling(60) avg pairwise equity corr (tail):
Date
2025-09-02    0.549627
2025-09-03    0.551725
2025-09-04    0.549764
2025-09-05    0.545939
2025-09-08    0.545741
2025-09-09    0.543770
2025-09-10    0.529330
2025-09-11    0.523707
2025-09-12    0.510802
2025-09-15    0.515767
dtype: float64
Ex-XMWX rolling(60) avg pairwise equity corr (tail):
Date
2024-12-02    0.426043
2024-12-03    0.410413
2024-12-04    0.410285
2024-12-05    0.417191
2024-12-06    0.403619
                ...   
2025-09-09    0.506077
2025-09-10    0.487784
2025-09-11    0.483393
2025-09-12    0.470275
2025-09-15    0.477077
Length: 192, dtype: float64


## 3-year view excluding XMWX
We will rebuild returns with a longer lookback (756 trading days ≈ 3y) and exclude `XMWX` so the window isn’t truncated, then recompute the 60D rolling average pairwise correlation.

## USD exposure check (manual)
This cell computes a per-holding USD exposure using explicit `USD_exposure` look-through where provided (e.g., WSML≈0.58), defaults to 1.0 for USD-priced assets not marked `risk_fx=='none'`, and includes `CASH_USD` as a negative hedge. Values are in CHF; a hedge recommendation in USD is derived from the `CASH_USD` series (USDCHF).